# persistance/02 — obstacle robustness *(behaviour + goal encoding)*

How robust is the model — and its goal encoding — when we throw an obstacle in the way at the
frontier? The **detour test** appends an adversarial prefill after the `{var} =` pivot and prints,
per example, the model's continuation (does it recover the correct op?) alongside the goal-head
read-out and affinity at the obstacle pivot (does the *encoding* still name the goal?).

> Scaling this to a proper obstacle-robustness eval (adversarial-prefill pass rate + encoding-persistence mediation) is TODO — see `agent_reports/claude/intro_reframing_plan.md` §4 and the brainstorm on the prior `incomplete_code.json` prefill-probing method.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, re, math, json, random, logging
sys.path.append("../..")                      # notebooks/persistance -> repo root
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "4,5")   # 32B ~63GB; set to free GPU(s)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import matplotlib.pyplot as plt
from rich.console import Console, Group
from rich.theme import Theme
from rich.table import Table
from rich.text import Text
from rich.box import SIMPLE_HEAVY
from rich.panel import Panel
from rich.markup import escape
from src.utils import logging_utils, env_utils
os.chdir(env_utils.PROJECT_ROOT); sys.path.insert(0, env_utils.PROJECT_ROOT)   # anchor CWD + imports to repo root so head_sets/ , data/ paths resolve
logging_utils.configure(level=logging.INFO)
logger = logging.getLogger(__name__)
console = Console(theme=Theme({"default": "white"}), highlight=False)   # highlight=False: don't green token strings
def rich_print(*a, **k): console.print(*a, **k)

## 0. Model · goal heads · encoding primitive

In [2]:
from src.models import ModelandTokenizer
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(model_key=MODEL_NAME, dtype=torch.bfloat16,
                       device_map="auto", attn_implementation="eager")
N_LAYER, N_HEADS = mt.n_layer, mt.n_heads
SLUG = MODEL_NAME.split("/")[-1]
print(f"n_layer={N_LAYER}  n_heads={N_HEADS}  n_embd={mt.n_embd}  head_dim={mt.head_dim}")

2026-09-20 16:17:18 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models
2026-09-20 16:17:18 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-20 16:17:18 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-09-20 16:17:18 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-20 16:17:18 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-09-20 16:17:18 httpx INFO     HTTP Re

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-09-20 16:17:45 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-20 16:17:45 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json "HTTP/1.1 200 OK"
2026-09-20 16:17:46 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0
n_layer=64  n_heads=40  n_embd=5120  head_dim=128


In [3]:
# ── goal heads (current JSON head sets — NOT the old head_mask.pt) ──
from scripts.evaluation.eval_goalscope import load_json_heads

HEAD_SETS = {
    250: f"head_sets/{SLUG}/goal_topk/goal__lam20_top250.json",
    122: f"head_sets/{SLUG}/goal__lam20_thr0.9_band25-56.json",
}
HEAD_BUDGET = 122                                   # <- toggle 250 / 122
goal_heads = load_json_heads(HEAD_SETS[HEAD_BUDGET], N_LAYER, N_HEADS)
print(f"{HEAD_BUDGET} goal heads · layers {min(l for l,_ in goal_heads)}-{max(l for l,_ in goal_heads)}")

122 goal heads · layers 28-55


In [4]:
# ── goal-encoding primitive (same as clock/14): the goal heads' summed residual write ──
from src.goals.head_loc_attn import collect_head_steering_vecs
from src.tokens import prepare_input

@torch.no_grad()
def goal_encoding(text, positions, heads=None, patch_interface=None):
    """(n_positions, n_embd) float32 — goal-head residual contribution at each position.
    `text` may be a str or a pre-tokenized TokenizerOutput (positions must match its tokenization)."""
    heads = goal_heads if heads is None else heads
    inp = prepare_input(prompts=text, tokenizer=mt) if isinstance(text, str) else text
    vecs = collect_head_steering_vecs(mt, inp, heads, token_positions=list(positions),
                                      patch_interface=patch_interface)
    return vecs[0]                                  # batch of 1 -> (n_pos, n_embd)

def cos(a, b):
    return float(torch.nn.functional.cosine_similarity(a.float(), b.float(), dim=-1))

## Corpus + helpers

In [5]:
# ── obstacle corpus: the CURRENT locked single-line-predicate eval set ──
from scripts.evaluation.eval_goalscope import (
    load_locked_datapoints, build_prompt, greedy_continue, frontier_impl_span, ALL_DATATYPES,
)
from src.goals.goalscope import apply_goalscope
from src.functional import generate_with_patch, PatchSpec

DATASET = "data/goalscope_eval.json"
DATATYPES = ["int", "str"]
dps, _ds_meta = load_locked_datapoints(DATASET, DATATYPES)   # returns (datapoints, meta) — unpack!
print(f"{len(dps)} datapoints across {DATATYPES}")

def clean_pivot(inst):
    """(prompt, var) ending at the frontier `{var} =` pivot — the committed-but-unwritten moment."""
    return build_prompt(mt, inst, "python")

def pivot_tok_positions(prompt):
    """token index of the final `=` pivot in a `… {var} =` prompt."""
    inp = prepare_input(prompts=prompt, tokenizer=mt); ids = inp.input_ids[0].tolist()
    return inp, len(ids) - 1                                # var_eq stub ends at `=`

def readout_at(text_or_inp, pos, gp=None):
    """GoalScope NL read-out at one position (single-shot)."""
    GS = dict(selected_heads=goal_heads, strategy="additive", alpha=0.2,
              steer_layer_range=(35, 46), n_steer_tokens="all", max_new_tokens=40)
    r = apply_goalscope(text_or_inp, mt=mt, show=False, include_noncode=False,
                        token_positions=[pos], goalscope_prompt=gp, **GS)
    return (r[0].get("patched_gen") or "").strip() if r else ""

129 datapoints across ['int', 'str']


## ① Detour test — obstacle at the frontier

Build the clean frontier prompt (`… {var} =`), then **inject an adversarial prefill** that pulls toward a
wrong op (e.g. `1 +`, `not`, `len(`). For 3–4 datapoints show:
- **behaviour:** clean continuation vs obstacle continuation (does it recover the correct op?),
- **encoding:** the goal-head read-out at the obstacle pivot (does it *still name the correct sub-goal*?)
  and the **affinity** cos(goal-encoding@obstacle, clean pivot encoding).

> Obstacles are improvised generic distractors; tune per predicate for a systematic run.

In [14]:
OBSTACLES = [" 1 +", " not", " 1 -"]      # generic distractors appended after `=`

def _with_pivot(inp, pos):
    """Full decoded text with the token at `pos` (the read-out extraction point) highlighted."""
    ids = inp.input_ids[0].tolist()
    return (f"{escape(mt.tokenizer.decode(ids[:pos]))}"
            f"[black on yellow]{escape(mt.tokenizer.decode([ids[pos]]))}[/black on yellow]"
            f"[yellow] ◂ read-out pivot[/yellow]")

def detour_example(inst):
    prompt, var = clean_pivot(inst)
    inp_clean, piv = pivot_tok_positions(prompt)
    v_clean = goal_encoding(inp_clean, [piv])[0]                    # clean TARGET_SIGNAL
    clean_gen = greedy_continue(mt, **inp_clean, max_new_tokens=32)
    ro_clean = readout_at(inp_clean, piv)

    obs = random.choice(OBSTACLES)
    obs_prompt = prompt + obs
    inp_obs = prepare_input(prompts=obs_prompt, tokenizer=mt)
    piv_obs = inp_obs.input_ids.shape[1] - 1                        # last obstacle token
    v_obs = goal_encoding(inp_obs, [piv_obs])[0]
    obs_gen = greedy_continue(mt, **inp_obs, max_new_tokens=32)
    ro_obs = readout_at(inp_obs, piv_obs)

    body = (
        f"[bold]target sub-goal:[/bold] {escape(inst.expected_next_desc)}\n\n"
        f"[bold sea_green3]── CLEAN ──[/bold sea_green3]  (read-out pivot = final `=`)\n"
        f"[grey58]prompt:[/grey58]\n{_with_pivot(inp_clean, piv)}\n"
        f"[grey58]generation:[/grey58] [sea_green3]{escape(clean_gen)}[/sea_green3]\n"
        f"[grey58]read-out @pivot:[/grey58] [sea_green3]{escape(ro_clean)}[/sea_green3]\n\n"
        f"[bold salmon1]── OBSTACLE ──[/bold salmon1]  (appended [bold]{escape(obs)!r}[/bold]"
        f"; read-out pivot = last obstacle token)\n"
        f"[grey58]prompt:[/grey58]\n{_with_pivot(inp_obs, piv_obs)}\n"
        f"[grey58]generation:[/grey58] [salmon1]{escape(obs_gen)}[/salmon1]\n"
        f"[grey58]read-out @pivot:[/grey58] [salmon1]{escape(ro_obs)}[/salmon1]\n\n"
        f"[bold]affinity cos(enc@obstacle-pivot, clean pivot) = {cos(v_obs, v_clean):+.3f}[/bold]"
    )
    return Panel(Text.from_markup(body), title=escape(inst.expected_next_desc),
                 border_style="grey42", padding=(0,1))

random.seed(0)
for _, inst in random.sample(dps, 6):
    rich_print(detour_example(inst))


GoalScope[additive]:   0%|          | 0/1 [00:00<?, ?it/s]

GoalScope[additive]:   0%|          | 0/1 [00:00<?, ?it/s]

╭──────────────────────────────────────── the number of uppercase letters ────────────────────────────────────────╮
│ target sub-goal: the number of uppercase letters                                                                │
│                                                                                                                 │
│ ── CLEAN ──  (read-out pivot = final `=`)                                                                       │
│ prompt:                                                                                                         │
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `func` which, given a string `s`, computes in sequential      │
│ order:                                                                                                          │
│ (1) whether the string has letters and all of them are lowercase                                                │
│ (2) the number of uppercase letters                                                                             │
│ (3) whether the string starts with an uppercase letter                                                          │
│ <|im_end|>                                                                                                      │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ def func(s: str):                                                                                               │
│     a = s.islower()                                                                                             │
│     b = ◂ read-out pivot                                                                                        │
│ generation:  sum(1 for c in s if c.isupper())                                                                   │
│     c = s[0].isupper() if s else False                                                                          │
│     return a, b                                                                                                 │
│ read-out @pivot: Sure, here's a quick breakdown:                                                                │
│                                                                                                                 │
│ ```python                                                                                                       │
│ sum(1 for c in s if c.isupper())                                                                                │
│ ```                                                                                                             │
│                                                                                                                 │
│ This line counts the number of uppercase letters in the string `s`.                                             │
│                                                                                                                 │
│ ── OBSTACLE ──  (appended ' not'; read-out pivot = last obstacle token)                                         │
│ prompt:                                                                                                         │
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming.

GoalScope[additive]:   0%|          | 0/1 [00:00<?, ?it/s]

GoalScope[additive]:   0%|          | 0/1 [00:00<?, ?it/s]

╭───────────────────────── whether the string has letters and all of them are lowercase ──────────────────────────╮
│ target sub-goal: whether the string has letters and all of them are lowercase                                   │
│                                                                                                                 │
│ ── CLEAN ──  (read-out pivot = final `=`)                                                                       │
│ prompt:                                                                                                         │
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `func` which, given a string `s`, computes in sequential      │
│ order:                                                                                                          │
│ (1) whether the first character is a letter                                                                     │
│ (2) whether the string has letters and all of them are lowercase                                                │
│ <|im_end|>                                                                                                      │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ def func(s: str):                                                                                               │
│     a = s[0].isalpha()                                                                                          │
│     b = ◂ read-out pivot                                                                                        │
│ generation:  s.isalpha() and s.islower()                                                                        │
│     return a, b                                                                                                 │
│ </code>                                                                                                         │
│ read-out @pivot: I'm checking if all characters in the string are alphabetic and if there is at least one       │
│ alphabetic character, using `str.isalpha()` and `any(c.isalpha() for c in s                                     │
│                                                                                                                 │
│ ── OBSTACLE ──  (appended ' not'; read-out pivot = last obstacle token)                                         │
│ prompt:                                                                                                         │
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `func` which, given a string `s`, computes in sequential      │
│ order:                                                                                                          │
│ (1) whether the first character is a letter                                                                     │
│ (2) whether the string has letters and all of them are

... (8 outputs left out)

╭────────────────────────────────────────────── the last character ───────────────────────────────────────────────╮
│ target sub-goal: the last character                                                                             │
│                                                                                                                 │
│ ── CLEAN ──  (read-out pivot = final `=`)                                                                       │
│ prompt:                                                                                                         │
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `func` which, given a string `s`, computes in sequential      │
│ order:                                                                                                          │
│ (1) the number of uppercase letters                                                                             │
│ (2) the last character                                                                                          │
│ (3) whether the string starts with an uppercase letter                                                          │
│ <|im_end|>                                                                                                      │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ def func(s: str):                                                                                               │
│     a = sum(1 for c in s if c.isupper())                                                                        │
│     b = ◂ read-out pivot                                                                                        │
│ generation:  s[-1] if s else ''                                                                                 │
│     c = s[0].isupper() if s else False                                                                          │
│     return a, b, c                                                                                              │
│ </                                                                                                              │
│ read-out @pivot: I'm looking at the last character of the string, which is `string[-1]`.                        │
│                                                                                                                 │
│ ── OBSTACLE ──  (appended ' 1 +'; read-out pivot = last obstacle token)                                         │
│ prompt:                                                                                                         │
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `func` which, given a string `s`, computes in sequential      │
│ order:                                                                                                          │
│ (1) the number of uppercase letters                   

GoalScope[additive]:   0%|          | 0/1 [00:00<?, ?it/s]

GoalScope[additive]:   0%|          | 0/1 [00:00<?, ?it/s]

╭──────────────────────────────────────── the number of lowercase letters ────────────────────────────────────────╮
│ target sub-goal: the number of lowercase letters                                                                │
│                                                                                                                 │
│ ── CLEAN ──  (read-out pivot = final `=`)                                                                       │
│ prompt:                                                                                                         │
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `func` which, given a string `s`, computes in sequential      │
│ order:                                                                                                          │
│ (1) the number of whitespace-separated words                                                                    │
│ (2) the number of lowercase letters                                                                             │
│ <|im_end|>                                                                                                      │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ def func(s: str):                                                                                               │
│     a = len(s.split())                                                                                          │
│     b = ◂ read-out pivot                                                                                        │
│ generation:  sum(1 for c in s if c.islower())                                                                   │
│     return a, b                                                                                                 │
│ </code>                                                                                                         │
│ read-out @pivot: I'm counting lowercase letters using a generator expression: `sum(c.islower() for c in s if    │
│ c.isalpha())`.                                                                                                  │
│                                                                                                                 │
│ ── OBSTACLE ──  (appended ' 1 -'; read-out pivot = last obstacle token)                                         │
│ prompt:                                                                                                         │
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `func` which, given a string `s`, computes in sequential      │
│ order:                                                                                                          │
│ (1) the number of whitespace-separated words                                                                    │
│ (2) the number of lowercase letters                   

### Notes / caveats
- The detour test is a **qualitative vibe-check** with improvised obstacles — for eyeballing, not
  numbers. The proper eval (graded adversarial prefills → unit-test/oracle pass rate, paired with
  encoding read-out so encoding-persistence can be tested as a *mediator* of behavioural recovery)
  is the next step; see the prior `data/incomplete_code.json` prefill-probing method
  (`agent_reports/claude/qualitative_analysis/0_lm_performance.md`) and
  `scripts/causality/adversarial_persistence.py`.